# Manual PyTorch Training Pipeline
This notebook implements a complete binary classification training pipeline in PyTorch **from scratch** (without using high-level modules like `nn.Module` or `torch.optim`).

We will perform:
1. Data preprocessing (loading, cleaning, splitting, scaling, and label encoding).
2. Custom tensor initialization for weights and biases.
3. Manual definition of the forward pass, Sigmoid activation, and Binary Cross-Entropy (BCE) loss.
4. A manual training loop utilizing PyTorch's automatic differentiation (`loss.backward()`) and gradient descent updates.

## 1. Importing Libraries

In [ ]:
# Importing libraries for preprocessing and numerical computations
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## 2. Loading the Dataset
We load the Breast Cancer Detection dataset directly from a raw CSV file.

In [ ]:
# Loading the dataset
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

## 3. Exploratory Data Analysis & Data Cleaning

In [ ]:
# Display initial dataset dimensions
print(f"No of rows : {df.shape[0]} ")
print(f"No of columns : {df.shape[1]} ")

In [ ]:
# Drop columns that are not predictive features (id and empty Unnamed: 32)
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [ ]:
# Display cleaned dataframe head
df.head()

In [ ]:
# Verify new dimensions
print(f"No of rows : {df.shape[0]} ")
print(f"No of columns : {df.shape[1]} ")

## 4. Preprocessing: Splitting and Feature Scaling
We split the dataset into training (80%) and testing (20%) sets, and scale features to have zero mean and unit variance.

In [ ]:
# Train-test split (Note: label is in column index 0; features in column index 1 onwards)
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

In [ ]:
# Scale the features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 5. Preprocessing: Label Encoding
We encode the categorical class labels (Malignant/Benign) to numerical labels (1/0).

In [ ]:
# View raw training labels before encoding
y_train

In [ ]:
# Fit and transform labels to binary integer targets
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
# View encoded training labels
y_train

## 6. Converting Data to PyTorch Tensors
We convert NumPy arrays to PyTorch Tensors so that gradients can be tracked.

In [ ]:
# Convert to PyTorch tensors (cast to float32/float64)
X_train_tensor = torch.from_numpy(X_train).double()
X_test_tensor = torch.from_numpy(X_test).double()
y_train_tensor = torch.from_numpy(y_train).double()
y_test_tensor = torch.from_numpy(y_test).double()

In [ ]:
# Inspect tensor shape of training features
X_train_tensor.shape

In [ ]:
# Inspect tensor shape of training labels
y_train_tensor.shape

## 7. Defining the Custom Single-Layer Neural Network
We define `MySimpleNN` containing manual weights, bias, forward propagation, and Binary Cross-Entropy loss computation.

In [ ]:
# Defining the model class from scratch
class MySimpleNN():
    def __init__(self, X):
        # Initialize weights randomly with require_grad=True to enable backpropagation
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        # Initialize bias to zeros with require_grad=True
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        # Linear transformation z = X * w + b
        z = torch.matmul(X, self.weights) + self.bias
        # Squash to [0, 1] range using Sigmoid function
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y):
        # Clamp predictions to avoid log(0) which results in NaN values
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        
        # Reshape y to (N, 1) to avoid PyTorch dimensions broadcasting bug (critical bug fix)
        y_reshaped = y.view(-1, 1)
        
        # Calculate Binary Cross-Entropy loss manually
        loss = -(y_reshaped * torch.log(y_pred) + (1 - y_reshaped) * torch.log(1 - y_pred)).mean()
        return loss

## 8. Training Configuration

In [ ]:
# Set learning rate and training epochs
learning_rate = 0.1
epochs = 25

## 9. Executing the Training Loop
We run the loop: compute forward pass -> calculate loss -> call `loss.backward()` to compute gradients -> perform manual gradient descent -> clear gradients manually for the next iteration.

In [ ]:
# Create the model instance
model = MySimpleNN(X_train_tensor)

# Start manual training loop
for epoch in range(epochs):
    # 1. Forward Pass
    y_pred = model.forward(X_train_tensor)

    # 2. Loss Calculation
    loss = model.loss_function(y_pred, y_train_tensor)

    # 3. Backward Pass (Computes gradients dLoss/dw and dLoss/db)
    loss.backward()

    # 4. Parameters Update (Using no_grad to prevent updating parameters tracking history)
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # 5. Reset Gradients manually (Important: PyTorch accumulates gradients by default)
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # Print loss value every epoch
    print(f'Epoch: {epoch + 1:02d} | Loss: {loss.item():.5f}')

## 10. Inspecting Final Learned Bias

In [ ]:
# Print the final bias parameter value
print("Final Bias:", model.bias.item())

## 11. Model Evaluation & Accuracy Measurement
We compute predictions on unseen test features and compare them to actual test targets to measure classification accuracy.

In [ ]:
# Model Evaluation
with torch.no_grad():
    # Generate prediction probabilities on the test set
    test_preds = model.forward(X_test_tensor)
    
    # Set decision threshold at 0.5 (classes squashed to binary classification predictions)
    binary_preds = (test_preds > 0.5).float()
    
    # Fix broadcasting issue by squeezing binary_preds shape from (N, 1) to (N,) to match y_test_tensor (critical bug fix)
    accuracy = (binary_preds.squeeze() == y_test_tensor).float().mean()
    
    print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')